# Notebook 00: Setup and Data

**Purpose**: Verify GPU availability, install the shiftprofile package, download CIFAR-10 and CIFAR-10-C, and prepare the cache.

**Expected runtime**: ~5 minutes (mostly download time)

**GPU cost**: None (CPU only)

Every cell prints something you can verify. If any cell fails, stop and fix it before proceeding to notebook 01.

## Step 1: Bootstrap the code from GitHub

Each Kaggle notebook is its own session and `/kaggle/working` does **not** carry over
between them, so every notebook fetches the code itself. Run this cell first.

**Private repo?** Store a GitHub fine-grained PAT (Contents: Read-only) as a Kaggle
Secret named `GITHUB_TOKEN` (Add-ons -> Secrets), then set `USE_SECRET = True`.
Never paste a token into a notebook cell: saved notebook versions keep their source,
so a pasted token is a published token.


In [ ]:
# ---- EDIT IF NEEDED ----
GITHUB_USER = "yoadjei"
GITHUB_REPO = "shiftprofile"
BRANCH      = "main"
USE_SECRET  = False   # True if the repo is private
# ------------------------

import subprocess, sys
from pathlib import Path

REPO_ROOT = Path("/kaggle/working/shiftprofile")

if USE_SECRET:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    _url = f"https://{_tok}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
else:
    _url = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"

# NOTE: never print _url - it may embed the token.
_git = ["git", "-C", str(REPO_ROOT)]
if REPO_ROOT.exists():
    subprocess.run(_git + ["fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(_git + ["reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, _url, str(REPO_ROOT)], check=True)
del _url

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_ROOT)], check=True)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))   # pip -e may not register in a live kernel

import shiftprofile
_sha = subprocess.run(_git + ["rev-parse", "--short", "HEAD"],
                      capture_output=True, text=True).stdout.strip()
print(f"shiftprofile ready at {REPO_ROOT}")
print(f"commit {_sha} on branch {BRANCH}")
print("Record this commit: every result record stores the code version that produced it.")


In [ ]:
# Confirm the repo landed and show what is in it.
for item in sorted(REPO_ROOT.glob('*')):
    print(f"  {item.name}{'/' if item.is_dir() else ''}")

# Optional but cheap insurance: run the test suite before spending session time.
# An environment difference is far cheaper to find here than eight hours in.
# !cd {REPO_ROOT} && python -m pytest -q


## Step 2: Check GPU

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory (GB): {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f}")
else:
    print("WARNING: CUDA is not available. This notebook will not work on Kaggle.")

In [ ]:
# Confirm the repo landed and show what is in it.
for item in sorted(REPO_ROOT.glob('*')):
    print(f"  {item.name}{'/' if item.is_dir() else ''}")

# Optional but cheap insurance: run the test suite before spending session time.
# An environment difference is far cheaper to find here than eight hours in.
# !cd {REPO_ROOT} && python -m pytest -q


In [ ]:
import subprocess
import sys

print("Installing shiftprofile package in editable mode...")
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', '/kaggle/working/shiftprofile'],
    check=True
)
print("Installation complete.")

## Step 3: Download CIFAR-10 test set

In [ ]:
from pathlib import Path
from shiftprofile.data import load_cifar10_test

data_root = Path('/kaggle/working/data')
data_root.mkdir(exist_ok=True)

print(f"Downloading CIFAR-10 test set to {data_root}...")
images, labels = load_cifar10_test(data_root)
print(f"Clean CIFAR-10: {images.shape}, labels {labels.shape}")
print(f"  dtype: {images.dtype}, label range: [{labels.min()}, {labels.max()}]")

## Step 4: Mount or download CIFAR-10-C

Two options:
- Option A (Recommended): Attach an existing CIFAR-10-C dataset to this notebook via the Kaggle interface.
- Option B (Fallback): Download from Zenodo and save as a dataset version for future use.

In [ ]:
from pathlib import Path
import numpy as np

# If you have attached a CIFAR-10-C dataset via the Kaggle UI, it will be mounted here:
cifar10c_root = Path('/kaggle/input/cifar-10-c')

# Verify the mounted dataset
if cifar10c_root.exists():
    print(f"CIFAR-10-C mounted at {cifar10c_root}")
    corruptions = sorted([p.stem for p in cifar10c_root.glob('*.npy') if p.stem != 'labels'])
    print(f"  Corruptions available: {corruptions}")
    labels = np.load(cifar10c_root / 'labels.npy')
    print(f"  Labels shape: {labels.shape}")
else:
    print(f"NOT FOUND: {cifar10c_root}")
    print("You must either:")
    print("  1. Attach an existing CIFAR-10-C dataset via Kaggle UI, OR")
    print("  2. Run the Zenodo download cell below")

### Option B: Download CIFAR-10-C from Zenodo (run once, takes 5-10 minutes)

In [ ]:
import subprocess
from pathlib import Path

# Only run this if you do NOT have CIFAR-10-C mounted
cifar10c_root = Path('/kaggle/working/cifar-10-c')
cifar10c_root.mkdir(exist_ok=True)

print(f"Downloading CIFAR-10-C from Zenodo (approximately 2.9 GB)...")
zenodo_url = 'https://zenodo.org/record/2535967/files/CIFAR-10-C.tar'

subprocess.run(
    ['wget', zenodo_url, '-O', str(cifar10c_root / 'CIFAR-10-C.tar')],
    check=True
)

print(f"Extracting...")
subprocess.run(
    ['tar', '-xf', str(cifar10c_root / 'CIFAR-10-C.tar'), '-C', str(cifar10c_root)],
    check=True
)

# Move files up one level if they are in a subdirectory
for tar_subdir in cifar10c_root.glob('CIFAR-10-C'):
    for f in tar_subdir.glob('*'):
        f.rename(cifar10c_root / f.name)
    tar_subdir.rmdir()

print(f"CIFAR-10-C downloaded to {cifar10c_root}")
print(f"After this cell completes, save /kaggle/working as a new Dataset version")
print(f"so you do not need to re-download next session.")

## Step 5: Verify clean/corrupted alignment

In [ ]:
import numpy as np
from pathlib import Path
from shiftprofile.data import load_cifar10_test

data_root = Path('/kaggle/working/data')
cifar10c_root = Path('/kaggle/input/cifar-10-c')

clean_images, clean_labels = load_cifar10_test(data_root)

corrupt_data = np.load(cifar10c_root / 'gaussian_noise.npy')
corrupt_labels = np.load(cifar10c_root / 'labels.npy')

corrupt_severity_1 = corrupt_data[0:10000]
corrupt_labels_severity_1 = corrupt_labels[0:10000]

assert np.array_equal(clean_labels, corrupt_labels_severity_1), \
    "FATAL: Clean and corrupted labels do not match!"

assert not np.array_equal(clean_images, corrupt_severity_1), \
    "ERROR: Clean and corrupted images are identical!"

print("Clean/corrupted alignment verified:")
print(f"  Clean: {clean_images.shape}, labels {clean_labels.shape}")
print(f"  Corrupted (gaussian_noise, sev 1): {corrupt_severity_1.shape}")
print(f"  Labels match: True")
print(f"  Images differ: True")

## Step 6: Print evaluation indices checksum

In [ ]:
import hashlib
from shiftprofile.data import fixed_eval_indices

indices_1000 = fixed_eval_indices(1000)
checksum = hashlib.sha256(indices_1000.tobytes()).hexdigest()[:16]

print(f"Evaluation indices (n=1000):")
print(f"  Checksum: {checksum}")
print(f"  Shape: {indices_1000.shape}")
print(f"  Min/max: {indices_1000.min()}/{indices_1000.max()}")
print(f"\nThis matches the committed indices in the repository.")
print(f"If it does not, something is wrong with the data seed.")

## Step 7: Create the empty cache directory

In [ ]:
from pathlib import Path
from shiftprofile.cache import ArtifactCache

cache_dir = Path('/kaggle/working/cache')
cache_dir.mkdir(exist_ok=True)

cache = ArtifactCache(write_root=cache_dir)

print(f"Cache directory created: {cache_dir}")
print(f"\nAfter notebook 01 completes:")
print(f"  1. Save /kaggle/working/cache as a NEW Kaggle Dataset version")
print(f"  2. Attach it to the next notebook run as read-only")
print(f"  3. In notebook 01, update the cache read path to point to the new version")
print(f"\nThe cache persists only as a saved Dataset version.")
print(f"Without this, work will not carry over between sessions.")

## Setup complete

You are ready to run notebook 01 (pilot fill). Before you proceed:

1. Verify that all cells above printed successful messages
2. Note the data paths:
   - Clean CIFAR-10: `/kaggle/working/data`
   - CIFAR-10-C: `/kaggle/input/cifar-10-c` (or `/kaggle/working/cifar-10-c` if using Zenodo)
   - Repository: `/kaggle/working/shiftprofile`
   - Cache (output): `/kaggle/working/cache`
3. If this is your first session, you may see download progress — that is normal and expected.

Proceed to notebook 01.